# Weather Whiplash — retrain the wetness probe (one-notebook pipeline)

Everything in one place: download **RSCD** (1M road patches, dry/wet/water)
and **RoadSaW** (CVPR-W 2022, wetness measured by a calibrated water-film
sensor) → extract a balanced subset in the exact folder format the trainer
needs → embed with frozen **CLIP** and **DINOv2** → train linear probes →
judge them with the **acceptance gates** against our 96 F1 frames.

WHY: leave-one-race-out on our own frames measured **36.5%** — five of six
races are single-class, so the old probe learned venue identity instead of
water. Thousands of road locations fix that by construction. The
architecture does not change: CLIP stays frozen, only the ~1,500-parameter
logistic regression is refit, and a winning CLIP probe is a **drop-in
replacement** for `backend/app/probe.npz`.

**How to run**
1. Runtime → Change runtime type → **GPU** (T4 is fine)
2. Run cells top to bottom. Cell 9 asks you to upload **calibrate.zip**
   (your `calibrate/` folder — `dry/ damp/ wet/` — zipped).
3. Every stage caches into `work_v2/`. If the runtime dies, re-run cells
   1–5 (fast — completed downloads no-op) and continue where you were.

**The deciding numbers** are the three gates in cell 9, not the validation
accuracy in cell 8 — roads validating against roads is the easy direction.
The F1 frames were never seen by these probes in any form:

```
gate 1   3-class accuracy on the 96 F1 frames   beats LORO 36.5%
gate 2   every rubbered-dry frame stays dry     the tyre-mark test
gate 3   Monaco damp < wet separation           >= 2.0 pooled SD
```

Licenses: RSCD **CC BY-NC**, RoadSaW **CC BY-NC-SA** — fine for the
hackathon, non-commercial beyond it.

In [ ]:
# 1) Environment
!nvidia-smi -L
!pip -q install "transformers>=5.0,<6.0" torch scikit-learn pillow numpy

In [ ]:
# 2) Configuration
from pathlib import Path
import numpy as np

WORK = Path("/content/work_v2")     # every stage caches its output here
PER_CLASS = 12000                   # images sampled per class per dataset
BACKBONES = ["clip", "dinov2"]      # the A/B arms
MONACO_KEY = "monaco2023"           # race prefix for gate 3

RSCD_URL = "https://figshare.com/ndownloader/files/36625041"   # 13.92 GB
# RoadSaW: 7.5 m distance / medium patch - the closest-range set, nearest
# to what our ROI crop sees. Other sets on the same host follow the pattern
# RoadSaW-{075,150,225,300}_{s,m,l}.zip - append URLs here to add them.
ROADSAW_URLS = [
    "http://downloads.viscoda.com/research/roadsaw/RoadSaW-075_m.zip",
]

# Production contract - backend/app/models/clip_scorer.py loads the .npz
# with exactly these keys and this class order; [0,50,100] is what the
# temporal layer's thresholds were tuned against. Do not reorder.
CLASSES = ["dry", "damp", "wet"]
VALUES = np.array([0.0, 50.0, 100.0])

# Baselines the gates compare against - the old probe's honest numbers.
BASE_3CLASS = 0.365      # leave-one-race-out, 3-class
BASE_WETNOT = 0.854      # wet vs not-wet, venue-held-out
BASE_MONACO_SD = 2.01    # damp->wet separation at Monaco

WORK.mkdir(parents=True, exist_ok=True)
IMAGES = WORK / "images"
PROBES = WORK / "probes"
RSCD_ZIP = WORK / "rscd_1million.zip"
ROADSAW_ZIPS = [WORK / u.rsplit("/", 1)[-1] for u in ROADSAW_URLS]
print("work dir:", WORK)

In [ ]:
# 3) Class mapping and zip subset extraction.
# Folder names inside the zips are matched with these rules. Anything
# UNMAPPED is printed loudly in cell 6 - edit the rule here if the actual
# naming differs; never silently drop data.
import hashlib, random, re, shutil, zipfile

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
SPLIT_NAMES = {"train", "val", "valid", "validation", "test"}


def map_rscd(folder):
    """RSCD friction labels -> our classes.

    RSCD 'wet' is a wet film with no standing water - our DAMP. RSCD
    'water' is standing water - our WET. Snow/ice do not occur on a race
    track in session; mud/gravel are not track surfaces.
    """
    n = folder.lower()
    if any(t in n for t in ("snow", "ice", "mud", "gravel", "dirt")):
        return None
    if "water" in n:
        return "wet"
    if "wet" in n:
        return "damp"
    if "dry" in n:
        return "dry"
    return None


def map_roadsaw(folder):
    """RoadSaW: 4 sensor-measured wetness levels -> our 3 classes.

    'very wet' merges into wet: production is 3-class and the FULL_WET
    distinction is made downstream by threshold. Numeric level codes
    (w0..w3) are a guess at the folder scheme - verify in cell 6's output.
    """
    n = folder.lower()
    if "very" in n and "wet" in n:
        return "wet"
    if "damp" in n or "moist" in n:
        return "damp"
    if "wet" in n:
        return "wet"
    if "dry" in n:
        return "dry"
    m = re.search(r"(?:^|[-_])w?([0-3])(?:$|[-_])", n)
    if m:
        return {0: "dry", 1: "damp", 2: "wet", 3: "wet"}[int(m.group(1))]
    return None


DATASET_RULES = {"rscd": map_rscd, "roadsaw": map_roadsaw}


def member_info(name, rule):
    """(mapped class | None, split) for one zip member path.

    Folders are tried innermost-first; the file stem is the last resort,
    because RoadSaW's internal layout is unverified and some datasets
    encode the label in the filename rather than the directory.
    """
    parts = [p for p in name.split("/") if p]
    split = next((p.lower() for p in parts if p.lower() in SPLIT_NAMES), "na")
    for part in reversed(parts[:-1]):
        cls = rule(part)
        if cls:
            return cls, split
    cls = rule(Path(parts[-1]).stem) if parts else None
    return cls, split


def extract_sample(zip_path, dataset, out_root, per_class, seed=0):
    """Stream a balanced sample straight out of the zip - the 1M-image
    archive is never fully unpacked."""
    rule = DATASET_RULES[dataset]
    by_class = {c: [] for c in CLASSES}
    unmapped = {}
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            if info.is_dir() or Path(info.filename).suffix.lower() not in IMG_EXT:
                continue
            cls, split = member_info(info.filename, rule)
            if cls is None:
                folder = Path(info.filename).parent.name
                unmapped[folder] = unmapped.get(folder, 0) + 1
                continue
            by_class[cls].append((info.filename, split))

        print()
        print(f"{dataset} ({zip_path.name}): mapped  " + "  ".join(
            f"{c}={len(by_class[c])}" for c in CLASSES))
        if unmapped:
            print(f"{dataset}: UNMAPPED folders - fix map_{dataset}() in"
                  f" cell 3 if any of these should count:")
            for k, v in sorted(unmapped.items(), key=lambda x: -x[1])[:20]:
                print(f"  {v:>8}  {k}")

        rng = random.Random(seed)
        n_out = 0
        for cls, members in by_class.items():
            rng.shuffle(members)
            dest_dir = out_root / dataset / cls
            dest_dir.mkdir(parents=True, exist_ok=True)
            for name, split in members[:per_class]:
                # The member-path hash keeps files distinct across source
                # folders; the split survives in the filename so training
                # can honour the official split later.
                h = hashlib.md5(name.encode()).hexdigest()[:8]
                dest = dest_dir / f"{split}__{h}_{Path(name).name}"
                with zf.open(name) as src, open(dest, "wb") as f:
                    shutil.copyfileobj(src, f)
                n_out += 1
        print(f"{dataset}: extracted {n_out} images")


print("mapping + extraction ready")

In [ ]:
# 4) Embedding - REPLICATES backend/app/models/clip_scorer.py exactly
# (same _as_embedding compat shim, same L2 normalisation). If you change
# one, change both.
import torch
from transformers import (AutoImageProcessor, AutoModel,
                          CLIPModel, CLIPProcessor)

BACKBONE_IDS = {"clip": "openai/clip-vit-base-patch32",
                "dinov2": "facebook/dinov2-small"}


def as_embedding(out, model, kind):
    """transformers v4 returns a tensor; v5 a BaseModelOutputWithPooling
    whose pooler_output is ALREADY projected. Never project blindly: on
    ViT-B/32 the text projection is 512->512, so double-projecting succeeds
    silently and corrupts the embedding - only the projection_dim
    comparison catches it."""
    if torch.is_tensor(out):
        return out
    for attr in ("text_embeds", "image_embeds"):
        val = getattr(out, attr, None)
        if torch.is_tensor(val):
            return val
    pooled = getattr(out, "pooler_output", None)
    if torch.is_tensor(pooled):
        target = getattr(model.config, "projection_dim", None)
        if target is None or pooled.shape[-1] == target:
            return pooled
        proj = model.text_projection if kind == "text" else model.visual_projection
        return proj(pooled)
    raise TypeError(f"cannot extract embedding from {type(out).__name__}")


class Embedder:
    """Batch image embedding for one backbone. GPU if available."""

    def __init__(self, backbone):
        self.backbone = backbone
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        hub_id = BACKBONE_IDS[backbone]
        if backbone == "clip":
            self.processor = CLIPProcessor.from_pretrained(hub_id)
            self.model = CLIPModel.from_pretrained(hub_id, use_safetensors=True)
        else:
            self.processor = AutoImageProcessor.from_pretrained(hub_id)
            self.model = AutoModel.from_pretrained(hub_id)
        self.model.eval().to(self.device)

    def embed(self, images):
        """L2-normalised embeddings for a list of PIL images."""
        with torch.no_grad():
            inputs = self.processor(images=images, return_tensors="pt")
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            if self.backbone == "clip":
                emb = as_embedding(self.model.get_image_features(**inputs),
                                   self.model, "image")
            else:
                out = self.model(**inputs)
                emb = getattr(out, "pooler_output", None)
                if emb is None:
                    emb = out.last_hidden_state[:, 0]
            emb = emb / emb.norm(dim=-1, keepdim=True)
            return emb.cpu().numpy().astype(np.float32)


print("embedder ready")

In [ ]:
# 5) Get the datasets. RoadSaW is small and direct. RSCD is 13.92 GB and
# figshare serves ONE slow stream - so the default route is the Kaggle
# mirror: Kaggle and Colab are both Google infrastructure, typically 10x
# faster. If the Kaggle route fails, set RSCD_SOURCE = "figshare" - that
# path uses aria2 with 16 parallel connections, far faster than wget.
import subprocess

RSCD_SOURCE = "kaggle"        # "kaggle" or "figshare"


def fetch(url, dest):
    # aria2: 16 parallel connections; -c resumes and no-ops when complete,
    # so this cell is safe to re-run at any point.
    subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"],
                   capture_output=True)
    r = subprocess.run(["aria2c", "-x16", "-s16", "-c",
                        "--console-log-level=warn",
                        "-d", str(dest.parent), "-o", dest.name, url])
    if r.returncode != 0:
        raise RuntimeError(f"download failed for {url} - re-run this cell,"
                           f" it resumes where it stopped")
    print(f"{dest.name}: {dest.stat().st_size / 1e9:.2f} GB")


for url, dest in zip(ROADSAW_URLS, ROADSAW_ZIPS):
    fetch(url, dest)

RSCD_DIR = None               # set if the mirror arrives pre-extracted
if RSCD_SOURCE == "kaggle":
    import kagglehub           # preinstalled on Colab; public sets need no login
    path = Path(kagglehub.dataset_download("cristvollerei/rscd-dataset-1million"))
    zips = sorted(path.rglob("*.zip"))
    if zips:
        RSCD_ZIP = zips[0]
        print("kaggle mirror zip:", RSCD_ZIP)
    else:
        RSCD_DIR = path        # kagglehub unpacked it already
        print("kaggle mirror pre-extracted at:", RSCD_DIR)
else:
    fetch(RSCD_URL, RSCD_ZIP)
print("datasets ready")

In [ ]:
# 6) Extract balanced subsets in the exact folder format the trainer
# needs:  work_v2/images/<dataset>/<class>/<split>__<hash>_<name>
#
# WATCH THE OUTPUT for UNMAPPED folders. RoadSaW's internal naming is
# unverified - if its folders land there, fix map_roadsaw() in cell 3 and
# re-run (delete work_v2/images/roadsaw first).
#
# A dataset whose download has not finished is SKIPPED, not fatal: you can
# run the whole pipeline on RoadSaW alone now and re-run cells 5-8 for RSCD
# later - cell 8 trains whatever datasets are present.


def extract_sample_dir(src_root, dataset, out_root, per_class, seed=0):
    """Same sampling as extract_sample, but from an already-extracted tree
    (the Kaggle mirror arrives unpacked)."""
    rule = DATASET_RULES[dataset]
    by_class = {c: [] for c in CLASSES}
    unmapped = {}
    for p in src_root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in IMG_EXT:
            continue
        rel = p.relative_to(src_root).as_posix()
        cls, split = member_info(rel, rule)
        if cls is None:
            unmapped[p.parent.name] = unmapped.get(p.parent.name, 0) + 1
            continue
        by_class[cls].append((p, rel, split))
    print()
    print(f"{dataset} ({src_root}): mapped  " + "  ".join(
        f"{c}={len(by_class[c])}" for c in CLASSES))
    if unmapped:
        print(f"{dataset}: UNMAPPED folders - fix map_{dataset}() in cell 3"
              f" if any of these should count:")
        for k, v in sorted(unmapped.items(), key=lambda x: -x[1])[:20]:
            print(f"  {v:>8}  {k}")
    rng = random.Random(seed)
    n_out = 0
    for cls, members in by_class.items():
        rng.shuffle(members)
        dest_dir = out_root / dataset / cls
        dest_dir.mkdir(parents=True, exist_ok=True)
        for p, rel, split in members[:per_class]:
            h = hashlib.md5(rel.encode()).hexdigest()[:8]
            shutil.copyfile(p, dest_dir / f"{split}__{h}_{p.name}")
            n_out += 1
    print(f"{dataset}: extracted {n_out} images")


RSCD_DIR = globals().get("RSCD_DIR", None)   # survives a skipped cell 5

if (IMAGES / "rscd").exists():
    print("rscd already extracted - skipping")
elif RSCD_DIR is not None:
    extract_sample_dir(RSCD_DIR, "rscd", IMAGES, PER_CLASS)
elif RSCD_ZIP.exists() and zipfile.is_zipfile(RSCD_ZIP):
    extract_sample(RSCD_ZIP, "rscd", IMAGES, PER_CLASS)
else:
    print("rscd zip missing or incomplete - SKIPPED. Continuing with"
          " RoadSaW only; re-run cells 5-8 once it lands.")

if (IMAGES / "roadsaw").exists():
    print("roadsaw already extracted - skipping")
else:
    for z in ROADSAW_ZIPS:
        if z.exists() and zipfile.is_zipfile(z):
            extract_sample(z, "roadsaw", IMAGES, PER_CLASS)
        else:
            print(f"{z.name} missing or incomplete - skipped")

for ds in ("rscd", "roadsaw"):
    d = IMAGES / ds
    if d.exists():
        counts = {c: len(list((d / c).glob("*"))) for c in CLASSES}
        print(ds, counts)
        missing = [c for c, n in counts.items() if n == 0]
        if missing:
            print(f"  ! {ds} has NO images for {missing} - a probe cannot"
                  f" learn a class it never sees")

In [ ]:
# 7) Embed every extracted image with each backbone (~10-20 min on a
# T4, cached - re-running skips finished backbones).
from PIL import Image


def collect(root):
    items = []
    for ds_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        for cls in CLASSES:
            cd = ds_dir / cls
            if not cd.is_dir():
                continue
            for p in sorted(cd.iterdir()):
                if p.suffix.lower() in IMG_EXT:
                    split = p.name.split("__", 1)[0] if "__" in p.name else "na"
                    items.append((p, ds_dir.name, cls, split))
    return items


items = collect(IMAGES)
assert items, "nothing under work_v2/images - run cell 6 first"
print(f"{len(items)} images, backbones: {BACKBONES}")
BATCH = 64
for backbone in BACKBONES:
    out = WORK / f"emb_{backbone}.npz"
    if out.exists():
        print(f"{backbone}: {out.name} exists - skipping")
        continue
    e = Embedder(backbone)
    print(f"{backbone} on {e.device}")
    X, keep, batch, meta = [], [], [], []
    for i, (p, ds, cls, split) in enumerate(items, 1):
        try:
            batch.append(Image.open(p).convert("RGB"))
            meta.append((ds, cls, split, p.name))
        except Exception as exc:
            print(f"  ! skip {p.name}: {exc}")
        if len(batch) >= BATCH or (i == len(items) and batch):
            X.append(e.embed(batch))
            keep.extend(meta)
            batch, meta = [], []
            done = sum(x.shape[0] for x in X)
            if done % (BATCH * 20) < BATCH:
                print(f"  {done}/{len(items)}")
    X = np.concatenate(X)
    np.savez_compressed(
        out, X=X,
        y=np.array([CLASSES.index(c) for _, c, _, _ in keep]),
        dataset=np.array([d for d, _, _, _ in keep]),
        split=np.array([s for _, _, s, _ in keep]),
        name=np.array([n for _, _, _, n in keep]))
    print(f"{backbone}: saved {X.shape} -> {out.name}")

In [ ]:
# 8) Train probes - one per backbone x data config (rscd / roadsaw /
# both). Validation accuracy here is a sanity number, NOT the decider.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

PROBES.mkdir(exist_ok=True)
SWEEP_CAP = 20000


def fit_probe(X_tr, y_tr, X_va, y_va, seed=0):
    """C sweep judged on the validation split, final fit on everything."""
    rng = np.random.RandomState(seed)
    if len(y_tr) > SWEEP_CAP:
        idx = rng.choice(len(y_tr), SWEEP_CAP, replace=False)
        Xs, ys = X_tr[idx], y_tr[idx]
    else:
        Xs, ys = X_tr, y_tr
    best = None
    print(f"  {'C':>8}{'val_acc':>10}")
    for C in (0.01, 0.1, 1.0, 10.0):
        clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced")
        clf.fit(Xs, ys)
        acc = accuracy_score(y_va, clf.predict(X_va))
        print(f"  {C:>8.2f}{acc:>10.3f}")
        if best is None or acc > best[0]:
            best = (acc, C)
    acc, C = best
    clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced")
    clf.fit(np.concatenate([X_tr, X_va]), np.concatenate([y_tr, y_va]))
    # If a class went missing the coef rows would silently misalign with
    # ['dry','damp','wet'] - refuse instead.
    assert list(clf.classes_) == [0, 1, 2], "a class is missing from training"
    return clf, C, acc


for backbone in BACKBONES:
    f = WORK / f"emb_{backbone}.npz"
    if not f.exists():
        print(f"no {f.name} - run cell 7")
        continue
    d = np.load(f, allow_pickle=True)
    X, y, dataset, split = d["X"], d["y"], d["dataset"], d["split"]
    present = sorted(set(dataset))
    configs = [(ds, dataset == ds) for ds in present]
    if len(present) > 1:
        configs.append(("both", np.ones(len(y), bool)))
    for cfg_name, mask in configs:
        out = PROBES / f"probe_{backbone}_{cfg_name}.npz"
        if out.exists():
            print(f"{out.name} exists - skipping")
            continue
        Xc, yc, sc = X[mask], y[mask], split[mask]
        if len(set(yc)) < 3:
            print(f"{backbone}/{cfg_name} lacks all 3 classes - skipped")
            continue
        # Honour the dataset's official split when it exists: RSCD frames
        # come from drives, so frames within a drive are near-siblings -
        # the exact leakage that once inflated our F1 number to 83.3%.
        va = np.isin(sc, ["val", "valid", "validation", "test"])
        if va.sum() < 100:
            va = np.random.RandomState(0).rand(len(yc)) < 0.2
            note = "random 20% - treat val_acc as optimistic; gates decide"
        else:
            note = "official split"
        print()
        print(f"train {backbone}/{cfg_name}  n={len(yc)}  "
              f"val={va.sum()}  ({note})")
        clf, C, acc = fit_probe(Xc[~va], yc[~va], Xc[va], yc[va])
        cm = confusion_matrix(yc[va], clf.predict(Xc[va]), labels=[0, 1, 2])
        for i, c in enumerate(CLASSES):
            print(f"    {c:>5}" + "".join(f"{v:>8}" for v in cm[i]))
        # Exact production keys; cv_accuracy is REQUIRED by the loader.
        np.savez(out, coef=clf.coef_, intercept=clf.intercept_,
                 classes=np.array(CLASSES), values=VALUES, C=C,
                 cv_accuracy=acc, backbone=backbone,
                 trained_on=cfg_name, n_train=len(yc))
        print(f"    saved {out.name}  (val_acc {acc:.3f})")

print()
print("trained:", sorted(p.name for p in PROBES.glob("probe_*.npz")))

In [ ]:
# 9) ACCEPTANCE GATES - the deciding numbers. The F1 frames were never
# seen by these probes in any form, so plain evaluation here is stricter
# than leave-one-race-out ever was for the old probe.
# Upload calibrate.zip (containing dry/ damp/ wet/) when prompted.
import os
from PIL import Image

F1_ZIP = Path("/content/calibrate.zip")
if not F1_ZIP.exists():
    from google.colab import files
    up = files.upload()          # choose calibrate.zip
    for name in up:
        if name != F1_ZIP.name:
            os.rename(name, str(F1_ZIP))

f1_root = WORK / "f1"
if not f1_root.exists():
    with zipfile.ZipFile(F1_ZIP) as zf:
        zf.extractall(f1_root)

# calibrate.zip may nest (calibrate/dry/...) - find class dirs anywhere.
f1_items = []
for cd in sorted(f1_root.rglob("*")):
    if cd.is_dir() and cd.name.lower() in CLASSES:
        for p in sorted(cd.iterdir()):
            if p.suffix.lower() in IMG_EXT:
                f1_items.append((p, CLASSES.index(cd.name.lower())))
assert f1_items, "no dry/damp/wet folders found inside calibrate.zip"
y_f1 = np.array([c for _, c in f1_items])
names_f1 = [p.name for p, _ in f1_items]
print(len(f1_items), "F1 frames:  " + "  ".join(
    f"{c}={int((y_f1 == i).sum())}" for i, c in enumerate(CLASSES)))


def race_of(name):
    m = re.match(r"([A-Za-z]+[0-9]{4})", Path(name).stem)
    return (m.group(1) if m else Path(name).stem.split("_")[0]).lower()


def run_gates(probe, X, y, names):
    """Same math as ClipScorer.score_with_probe, vectorised."""
    logits = X @ probe["coef"].T + probe["intercept"]
    z = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(z)
    probs /= probs.sum(axis=1, keepdims=True)
    pred = probs.argmax(axis=1)
    wet = probs @ np.asarray(probe["values"], dtype=float)
    dry = y == 0
    races = np.array([race_of(n) for n in names])
    a = wet[(races == MONACO_KEY) & (y == 1)]
    b = wet[(races == MONACO_KEY) & (y == 2)]
    if len(a) and len(b):
        pooled = np.sqrt((a.std() ** 2 + b.std() ** 2) / 2) or 1e-9
        sd = float(abs(b.mean() - a.mean()) / pooled)
    else:
        sd = float("nan")
    return {
        "acc3": float((pred == y).mean()),
        "accw": float(((pred == 2) == (y == 2)).mean()),
        "dry_ok": float((pred[dry] == 0).mean()) if dry.any() else float("nan"),
        "offenders": [(names[i], float(wet[i]))
                      for i in np.where(dry & (pred != 0))[0]],
        "monaco_sd": sd,
    }


def mark(ok):
    return "PASS" if ok else "FAIL"


cache = {}
results = []
for probe_path in sorted(PROBES.glob("probe_*.npz")):
    d = np.load(probe_path, allow_pickle=True)
    probe = {k: d[k] for k in ("coef", "intercept", "values")}
    backbone = str(d["backbone"])
    if backbone not in cache:
        e = Embedder(backbone)
        cache[backbone] = np.stack(
            [e.embed([Image.open(p).convert("RGB")])[0] for p, _ in f1_items])
    g = run_gates(probe, cache[backbone], y_f1, names_f1)
    results.append((probe_path.name, g))
    print()
    print(probe_path.name)
    print(f"  gate 1  3-class {g['acc3']:.3f} vs LORO {BASE_3CLASS:.3f}  "
          f"[{mark(g['acc3'] > BASE_3CLASS)}]   "
          f"wet/not-wet {g['accw']:.3f} vs {BASE_WETNOT:.3f}")
    print(f"  gate 2  dry frames kept dry {g['dry_ok']:.1%}  "
          f"[{mark(g['dry_ok'] >= 0.90)}]  (tyre-mark test)")
    for n, w in g["offenders"][:8]:
        print(f"            miss: {n}  wetness {w:.1f}")
    ok3 = not np.isnan(g["monaco_sd"]) and g["monaco_sd"] >= 2.0
    print(f"  gate 3  monaco damp->wet {g['monaco_sd']:.2f} SD vs "
          f"{BASE_MONACO_SD:.2f}  [{mark(ok3)}]")

results.sort(key=lambda r: (r[1]["acc3"], r[1]["dry_ok"]), reverse=True)
best, g = results[0]
print()
print(f"ranked winner: {best}  (3-class {g['acc3']:.3f})")
if "clip" in best:
    print("CLIP backbone -> DROP-IN: copy it over backend/app/probe.npz"
          " and restart the server.")
else:
    print("DINOv2 backbone -> NOT drop-in (the backend embeds with CLIP);"
          " a small backend change is needed before it can ship.")

In [ ]:
# 10) Package and download the probes.
# On the laptop: unzip, copy the winning CLIP probe over
# backend\app\probe.npz and restart uvicorn - the loader prints the new
# accuracy on startup and /health reports it.
!cd {WORK} && zip -q -r /content/probes_v2.zip probes
!ls -la /content/probes_v2.zip
from google.colab import files
files.download("/content/probes_v2.zip")